# 急诊合成临床数据审计与固定划分

本笔记本读取Kaggle官方六个CSV，核验记录、字段和关联键，并生成研究计划约定的固定分区。开发阶段的标签统计仅针对D。输出表采用三线表，图表题名位于图表下方。

## 设置与来源

运行前激活machinelearning环境。配置位于configs/research_protocol.json。原始数据来源和文件哈希随本次输出保存。

In [ ]:
from collections import Counter
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import hashlib
import json
import os

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
    StratifiedShuffleSplit,
)

PROJECT_ROOT = Path(os.environ.get(
    "ML_PROJECT_ROOT",
    ".",
)).resolve()
config_path = Path(os.environ.get(
    "ML_EXPERIMENT_CONFIG",
    str(PROJECT_ROOT / "configs/research_protocol.json"),
))
config = json.loads(config_path.read_text(encoding="utf-8"))
raw_directory = PROJECT_ROOT / config["raw_directory"]
processed_directory = PROJECT_ROOT / config["processed_directory"]
audit_root = PROJECT_ROOT / config["audit_directory"]
run_id = datetime.now(ZoneInfo("Asia/Shanghai")).strftime("%Y%m%dT%H%M%S")
audit_directory = audit_root / run_id
audit_directory.mkdir(parents=True, exist_ok=False)
processed_directory.mkdir(parents=True, exist_ok=True)
identifier = config["id_column"]
targets = config["target_columns"]
random_seed = config["random_seed"]
audit_events = []
print("配置版本", config["protocol_version"])
print("数据类型", config["dataset_kind"])
print("本次输出", audit_directory)

## 文件与模式检查

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def save_json(filename, value):
    destination = audit_directory / filename
    destination.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

def scientific_table(frame, caption, precision=4):
    styled = (
        frame.style
        .hide(axis="index")
        .format(precision=precision, na_rep="—")
        .set_caption(caption)
        .set_table_styles([
            {"selector": "", "props": [
                ("border-collapse", "collapse"),
                ("border-top", "1.5pt solid black"),
                ("border-bottom", "1.5pt solid black"),
                ("font-family", "'Times New Roman', SimSun, serif"),
            ]},
            {"selector": "thead th", "props": [
                ("border-bottom", "0.75pt solid black"),
                ("border-left", "none"),
                ("border-right", "none"),
                ("background-color", "white"),
                ("color", "black"),
                ("padding", "6px 10px"),
            ]},
            {"selector": "tbody td", "props": [
                ("border", "none"),
                ("padding", "5px 10px"),
                ("background-color", "white"),
                ("color", "black"),
            ]},
            {"selector": "caption", "props": [
                ("caption-side", "bottom"),
                ("text-align", "center"),
                ("color", "black"),
                ("padding-top", "7px"),
            ]},
        ])
    )
    display(HTML(styled.to_html()))

missing_files = [
    name for name in config["expected_files"]
    if not (raw_directory / name).is_file()
]
if missing_files:
    raise FileNotFoundError("尚缺少官方原始文件 " + ", ".join(missing_files))

file_records = []
frames = {}
for filename in config["expected_files"]:
    path = raw_directory / filename
    import csv
    with path.open("r", encoding="utf-8-sig", newline="") as header_stream:
        raw_header = next(csv.reader(header_stream))
    if len(raw_header) != len(set(raw_header)):
        raise ValueError("原始CSV具有重复列名 " + filename)
    frame = pd.read_csv(path, dtype_backend="pyarrow")
    if frame.columns.duplicated().any():
        raise ValueError("存在重复列名 " + filename)
    frames[filename] = frame
    expected = config["expected_rows"].get(filename)
    file_records.append({
        "file": filename,
        "rows": len(frame),
        "columns": len(frame.columns),
        "bytes": path.stat().st_size,
        "sha256": file_sha256(path),
        "expected_rows_match": expected is None or len(frame) == expected,
    })
    if expected is not None and len(frame) != expected:
        audit_events.append({
            "issue": "row_count_differs_from_public_description",
            "source": filename,
            "actual": len(frame),
            "expected": expected,
        })
file_inventory = pd.DataFrame(file_records)
save_json("source_files.json", file_records)
scientific_table(
    file_inventory[["file", "rows", "columns", "bytes", "expected_rows_match"]],
    "表1 原始输入文件与公开规模的核对",
    precision=0,
)
for filename in ["train.csv", "supplemental_data.csv"]:
    required = {identifier, *targets}
    missing = required - set(frames[filename].columns)
    if missing:
        raise ValueError(f"{filename} 缺少必需列 {sorted(missing)}")
for filename in ["test.csv", "sample_submission.csv"]:
    if identifier not in frames[filename].columns:
        raise ValueError(f"{filename} 缺少就诊标识")
if set(frames["test.csv"][identifier]) != set(frames["sample_submission.csv"][identifier]):
    raise ValueError("测试集与提交模板的ID集合不一致")
if frames["sample_submission.csv"][identifier].duplicated().any():
    raise ValueError("提交模板含重复ID")
train_columns = set(frames["train.csv"].columns)
test_columns = set(frames["test.csv"].columns)
save_json("schema.json", {
    name: {column: str(dtype) for column, dtype in frame.dtypes.items()}
    for name, frame in frames.items()
})
save_json("schema_differences.json", {
    "train_only": sorted(train_columns - test_columns),
    "test_only": sorted(test_columns - train_columns),
})

## 记录有效性与跨文件重叠

In [ ]:
def audited_records(frame, source_name, labeled):
    frame = frame.copy()
    original_count = len(frame)
    id_valid = frame[identifier].notna()
    frame[identifier] = frame[identifier].astype("string").str.strip()
    id_valid = id_valid & frame[identifier].ne("").fillna(False)
    invalid_label = pd.Series(False, index=frame.index)
    if labeled:
        for target in targets:
            invalid_label |= ~frame[target].isin([0, 1])
    invalid = ~id_valid | invalid_label
    excluded = frame.loc[invalid, [identifier]].copy()
    excluded["reason"] = np.where(
        ~id_valid.loc[invalid].to_numpy(), "invalid_id", "invalid_target"
    )
    frame = frame.loc[~invalid].copy()
    exact_duplicate = frame.duplicated(keep="first")
    duplicate_count = int(exact_duplicate.sum())
    frame = frame.loc[~exact_duplicate].copy()
    conflicting = frame[identifier].duplicated(keep=False)
    conflicting_ids = frame.loc[conflicting, [identifier]].copy()
    conflicting_ids["reason"] = "conflicting_duplicate_id"
    excluded = pd.concat([excluded, conflicting_ids], ignore_index=True)
    frame = frame.loc[~conflicting].reset_index(drop=True)
    excluded.to_csv(
        audit_directory / (Path(source_name).stem + "_excluded.csv"),
        index=False,
    )
    audit_events.append({
        "source": source_name,
        "original_rows": original_count,
        "exact_duplicate_rows_removed": duplicate_count,
        "conflicting_duplicate_rows_removed": int(conflicting.sum()),
        "invalid_rows_removed": int(invalid.sum()),
        "remaining_rows": len(frame),
    })
    return frame

train = audited_records(frames["train.csv"], "train.csv", labeled=True)
supplemental = audited_records(
    frames["supplemental_data.csv"], "supplemental_data.csv", labeled=True
)
test = frames["test.csv"].copy()
if test[identifier].isna().any() or test[identifier].duplicated().any():
    raise ValueError("测试ID无效，保留原始文件供核查")
test[identifier] = test[identifier].astype("string").str.strip()
if test[identifier].eq("").any() or test[identifier].duplicated().any():
    raise ValueError("测试ID规范化后存在空值或重复")
test_ids = set(test[identifier])
train_overlap = train[identifier].isin(test_ids)
audit_events.append({
    "issue": "train_test_id_overlap_removed_from_training",
    "rows": int(train_overlap.sum()),
})
train = train.loc[~train_overlap].reset_index(drop=True)
all_main_ids = set(train[identifier]) | test_ids
supplemental_overlap = supplemental[identifier].isin(all_main_ids)
audit_events.append({
    "issue": "supplemental_main_id_overlap",
    "rows": int(supplemental_overlap.sum()),
})
supplemental = supplemental.loc[~supplemental_overlap].reset_index(drop=True)

year_column = config["year_column"]
if year_column is None:
    matches = [name for name in config["year_candidates"] if name in train.columns]
    if len(matches) != 1:
        save_json("audit_events.json", audit_events)
        raise ValueError("需要依据实际字段字典确定年份列 " + str(matches))
    year_column = matches[0]
patient_column = config["patient_id_column"]
if patient_column is None:
    matches = [
        name for name in config["patient_id_candidates"]
        if name in train.columns and name in supplemental.columns
    ]
    if len(matches) > 1:
        raise ValueError("患者标识候选不唯一")
    patient_column = matches[0] if matches else None
years = pd.to_numeric(train[year_column], errors="coerce")
development_mask = years.isin(config["development_years"])
temporal_mask = years.isin(config["temporal_test_years"])
outside_time = ~(development_mask | temporal_mask)
audit_events.append({
    "issue": "invalid_or_outside_protocol_year",
    "rows": int(outside_time.sum()),
})
development = train.loc[development_mask].reset_index(drop=True)
temporal_test = train.loc[temporal_mask].reset_index(drop=True)
if development.empty or temporal_test.empty:
    raise ValueError("预定开发集或时间测试集为空")

if patient_column:
    datasets_with_patient = [development, temporal_test, supplemental]
    if any(frame[patient_column].isna().any() for frame in datasets_with_patient):
        save_json("audit_events.json", audit_events)
        raise ValueError("已有患者标识列包含缺失，需要先确定患者分组处理规则")
    for frame in datasets_with_patient:
        frame[patient_column] = frame[patient_column].astype("string")
    future_patients = set(temporal_test[patient_column])
    past_overlap = development[patient_column].isin(future_patients)
    audit_events.append({
        "issue": "development_patients_overlapping_temporal_test",
        "rows": int(past_overlap.sum()),
    })
    development = development.loc[~past_overlap].reset_index(drop=True)
    main_patients = set(development[patient_column]) | future_patients
    repeated_patients = supplemental[patient_column].isin(main_patients)
    audit_events.append({
        "issue": "supplemental_patients_overlapping_main",
        "rows": int(repeated_patients.sum()),
    })
    supplemental = supplemental.loc[~repeated_patients].reset_index(drop=True)
else:
    audit_events.append({
        "issue": "patient_identifier_not_available",
        "interpretation": "隔离与重抽样只能以就诊记录为单位",
    })
save_json("audit_events.json", audit_events)
partition_counts = pd.DataFrame([
    {"partition": "D development", "rows": len(development)},
    {"partition": "H temporal test", "rows": len(temporal_test)},
    {"partition": "S supplemental", "rows": len(supplemental)},
    {"partition": "K Kaggle test", "rows": len(test)},
])
scientific_table(partition_counts, "表2 数据审计后的分区规模", precision=0)
print("年份列", year_column)
print("患者标识列", patient_column or "未提供")

## 开发数据概况与参考表完整性

此处标签统计仅使用开发集D。机构参考表按city核对唯一性和覆盖，不在审计中产生结局编码。

In [ ]:
label_records = []
for target in targets:
    if development[target].nunique() != 2:
        raise ValueError("开发目标缺少两个有效类别 " + target)
    positives = int(development[target].sum())
    label_records.append({
        "target": target,
        "development_rows": len(development),
        "positive_rows": positives,
        "positive_fraction": positives / len(development),
    })
scientific_table(pd.DataFrame(label_records), "表3 开发集两个目标的标签分布")
save_json("development_labels.json", label_records)

excluded_predictor_columns = {identifier, *targets, "disposition"}
feature_columns = [
    name for name in development.columns
    if name not in excluded_predictor_columns and name != patient_column
]
missing_profile = pd.DataFrame({
    "column": feature_columns,
    "missing_rows": [
        int(development[column].isna().sum()) for column in feature_columns
    ],
    "missing_fraction": [
        float(development[column].isna().mean()) for column in feature_columns
    ],
    "distinct_values": [
        int(development[column].nunique(dropna=True)) for column in feature_columns
    ],
    "dtype": [str(development[column].dtype) for column in feature_columns],
}).sort_values(
    ["missing_fraction", "column"], ascending=[False, True]
).reset_index(drop=True)
missing_profile.to_csv(audit_directory / "development_column_profile.csv", index=False)
scientific_table(missing_profile.head(20), "表4 开发集缺失比例最高的20个字段")

reference = frames["hospital_reference.csv"]
if "city" not in reference.columns or "city" not in train.columns:
    raise ValueError("机构参考关联键city缺失，需要复核数据模式")
if reference["city"].isna().any() or reference["city"].duplicated().any():
    raise ValueError("机构参考表的city不是有效唯一键")
unmatched_cities = sorted(
    str(value) for value in set(train["city"].dropna()) - set(reference["city"])
)
save_json("reference_integrity.json", {
    "key": "city",
    "reference_rows": len(reference),
    "unique_key": True,
    "unmatched_cities": unmatched_cities,
    "hypothetical_left_join_preserves_rows": not reference["city"].duplicated().any(),
})
if unmatched_cities:
    raise ValueError("存在未匹配的城市，需要先复核参考表")
print("机构参考关联检查通过")

## 固定外层五折与内部训练角色

外层V用于开发评价，T内部划出F和C。F内部三折只为学习融合权重生成折外预测。有患者标识时，各层都保持患者分组。

In [ ]:
def stratification_labels(frame, minimum_count):
    joint = (
        frame[targets[0]].astype("int8") * 2
        + frame[targets[1]].astype("int8")
    )
    if joint.value_counts().min() >= minimum_count:
        return joint.to_numpy(), "joint_targets"
    primary = frame[targets[0]].astype("int8")
    if primary.value_counts().min() < minimum_count:
        raise ValueError("目标A的类别规模不足以执行预定划分")
    return primary.to_numpy(), "adverse_outcome"

def fold_assignments(frame, fold_count, seed):
    labels, basis = stratification_labels(frame, fold_count)
    positions = np.zeros((len(frame), 1), dtype=np.int8)
    assignments = np.full(len(frame), -1, dtype=np.int8)
    if patient_column:
        groups = frame[patient_column].to_numpy()
        splitter = StratifiedGroupKFold(
            n_splits=fold_count, shuffle=True, random_state=seed
        )
        splits = splitter.split(positions, labels, groups=groups)
    else:
        splitter = StratifiedKFold(
            n_splits=fold_count, shuffle=True, random_state=seed
        )
        splits = splitter.split(positions, labels)
    for fold, (training_positions, validation_positions) in enumerate(splits):
        assignments[validation_positions] = fold
        if patient_column:
            training_groups = set(frame.iloc[training_positions][patient_column])
            validation_groups = set(frame.iloc[validation_positions][patient_column])
            if training_groups & validation_groups:
                raise AssertionError("患者跨折重叠")
    if (assignments < 0).any():
        raise AssertionError("存在未分配记录")
    return assignments, basis

split_columns = [identifier, year_column, *targets]
if patient_column:
    split_columns.append(patient_column)
split_frame = development[split_columns].copy()
outer_assignments, outer_basis = fold_assignments(
    split_frame, config["outer_folds"], random_seed
)
manifest = split_frame[[identifier, year_column]].copy()
if patient_column:
    manifest[patient_column] = split_frame[patient_column]
manifest["outer_fold"] = outer_assignments
role_records = []
for outer_fold in range(config["outer_folds"]):
    training_positions = np.flatnonzero(outer_assignments != outer_fold)
    validation_positions = np.flatnonzero(outer_assignments == outer_fold)
    training_frame = split_frame.iloc[training_positions].reset_index(drop=True)
    labels, calibration_basis = stratification_labels(
        training_frame, config["outer_folds"] if patient_column else 2
    )
    if patient_column:
        calibration_splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=random_seed + outer_fold
        )
        fitting_local, calibration_local = next(calibration_splitter.split(
            np.zeros((len(training_frame), 1), dtype=np.int8),
            labels,
            groups=training_frame[patient_column].to_numpy(),
        ))
    else:
        calibration_splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=config["calibration_fraction"],
            random_state=random_seed + outer_fold,
        )
        fitting_local, calibration_local = next(calibration_splitter.split(
            np.zeros((len(training_frame), 1), dtype=np.int8), labels
        ))
    fitting_positions = training_positions[fitting_local]
    calibration_positions = training_positions[calibration_local]
    roles = np.full(len(split_frame), "V", dtype="U1")
    roles[fitting_positions] = "F"
    roles[calibration_positions] = "C"
    inner_assignments, inner_basis = fold_assignments(
        split_frame.iloc[fitting_positions].reset_index(drop=True),
        config["inner_folds"],
        random_seed + outer_fold,
    )
    inner_column = np.full(len(split_frame), -1, dtype=np.int8)
    inner_column[fitting_positions] = inner_assignments
    manifest[f"role_outer_{outer_fold}"] = pd.Categorical(
        roles, categories=["F", "C", "V"]
    )
    manifest[f"inner_fold_outer_{outer_fold}"] = inner_column
    if patient_column:
        group_sets = [
            set(split_frame.iloc[positions][patient_column])
            for positions in [
                fitting_positions, calibration_positions, validation_positions
            ]
        ]
        if any(
            group_sets[left] & group_sets[right]
            for left, right in [(0, 1), (0, 2), (1, 2)]
        ):
            raise AssertionError("F/C/V之间存在患者重叠")
    role_records.append({
        "outer_fold": outer_fold,
        "fitting_rows": len(fitting_positions),
        "calibration_rows": len(calibration_positions),
        "validation_rows": len(validation_positions),
        "outer_stratification": outer_basis,
        "calibration_stratification": calibration_basis,
        "inner_stratification": inner_basis,
    })
scientific_table(
    pd.DataFrame(role_records)[[
        "outer_fold", "fitting_rows", "calibration_rows", "validation_rows"
    ]],
    "表5 外层五折中F、C、V的实际规模",
    precision=0,
)
save_json("fold_counts.json", role_records)

## 保存审计版本与固定数据清单

每个发布版本由原始训练文件哈希和配置哈希标识。正式训练按固定清单选择开发记录，时间测试结果在方案锁定后计算。

In [ ]:
import nbformat

config_digest = hashlib.sha256(
    json.dumps(config, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()
notebook_path = PROJECT_ROOT / "notebooks/01_data_audit.ipynb"
notebook_definition = nbformat.read(notebook_path, as_version=4)
implementation_digest = hashlib.sha256(
    "\n".join(
        cell.source for cell in notebook_definition.cells
        if cell.cell_type == "code"
    ).encode("utf-8")
).hexdigest()
training_digest = next(
    item["sha256"] for item in file_records if item["file"] == "train.csv"
)
source_digest = hashlib.sha256(
    json.dumps({item["file"]: item["sha256"] for item in file_records}, sort_keys=True).encode("utf-8")
).hexdigest()
release_id = (
    source_digest[:12] + "_" + config_digest[:8] + "_"
    + implementation_digest[:8]
)
release_directory = processed_directory / release_id
release_manifest_directory = release_directory / config["manifest_subdirectory"]
pointer_path = PROJECT_ROOT / config["active_release_pointer"]
if pointer_path.exists():
    active_pointer = json.loads(pointer_path.read_text(encoding="utf-8"))
    if active_pointer["release_id"] != release_id:
        raise RuntimeError("已有不同的活动数据版本，需要先核验版本切换")

datasets_to_save = {
    "development": development,
    "temporal_test_locked": temporal_test,
    "supplemental_locked": supplemental,
    "kaggle_test": test,
}
partition_manifest_records = []
for name, frame in datasets_to_save.items():
    values = frame[[identifier]].copy()
    values["partition"] = name
    partition_manifest_records.append(values)
partition_manifest = pd.concat(partition_manifest_records, ignore_index=True)
if partition_manifest[identifier].duplicated().any():
    raise AssertionError("分区间存在重复就诊标识")
release = {
    "release_id": release_id,
    "dataset_kind": config["dataset_kind"],
    "source_url": config["source_url"],
    "protocol_version": config["protocol_version"],
    "config_sha256": config_digest,
    "implementation_sha256": implementation_digest,
    "source_files": file_records,
    "source_bundle_sha256": source_digest,
    "year_column": year_column,
    "patient_id_column": patient_column,
    "grouping_unit": "patient" if patient_column else "encounter",
    "partition_counts": partition_counts.to_dict(orient="records"),
    "processed_directory": str(release_directory),
    "manifest_directory": str(release_manifest_directory),
    "development_labels": label_records,
    "fold_counts": role_records,
    "audit_directory": str(audit_directory),
    "temporal_test_target_statistics": "reserved_for_locked_evaluation",
}
if release_directory.exists():
    existing_release = json.loads(
        (release_directory / "release.json").read_text(encoding="utf-8")
    )
    for relative_name, expected_hash in existing_release["artifact_sha256"].items():
        if file_sha256(release_directory / relative_name) != expected_hash:
            raise ValueError("已发布数据文件校验失败 " + relative_name)
    release = existing_release
    print("复用已校验的固定数据版本", release_id)
else:
    staging_directory = processed_directory / (".staging_" + release_id + "_" + run_id)
    staging_directory.mkdir(exist_ok=False)
    staging_manifests = staging_directory / config["manifest_subdirectory"]
    staging_manifests.mkdir()
    for name, frame in datasets_to_save.items():
        frame.to_parquet(staging_directory / f"{name}.parquet", index=False)
    manifest.to_parquet(staging_manifests / "development_folds.parquet", index=False)
    partition_manifest.to_parquet(staging_manifests / "partitions.parquet", index=False)
    release["artifact_sha256"] = {
        str(path.relative_to(staging_directory)): file_sha256(path)
        for path in sorted(staging_directory.rglob("*.parquet"))
    }
    (staging_directory / "release.json").write_text(
        json.dumps(release, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    staging_directory.rename(release_directory)
    print("已发布固定数据版本", release_id)
pointer = {
    "release_id": release_id,
    "release_json": str(release_directory / "release.json"),
}
if not pointer_path.exists():
    temporary_pointer = pointer_path.with_name(
        ".active_data_release_" + run_id + ".json"
    )
    temporary_pointer.write_text(
        json.dumps(pointer, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary_pointer.replace(pointer_path)
save_json("release.json", release)

## 审计图形与数据解释

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
})
resolved_font = font_manager.FontProperties(
    fname=font_manager.findfont(font_manager.FontProperties(family="serif"))
).get_name()

def save_audit_figure(figure, filename, caption):
    figure.text(0.5, 0.02, caption, ha="center", va="bottom", fontsize=10)
    figure.savefig(
        audit_directory / filename, dpi=200,
        bbox_inches="tight", facecolor="white"
    )
    display(figure)
    plt.close(figure)

figure, axis = plt.subplots(figsize=(7.2, 4.4))
positions = np.arange(len(partition_counts))
axis.bar(
    positions, partition_counts["rows"], color="#57738C",
    edgecolor="black", linewidth=0.6
)
axis.set_xticks(positions, ["D", "H", "S", "K"])
axis.set_xlabel("Protocol partition")
axis.set_ylabel("Encounters")
axis.set_ylim(bottom=0)
for position, count in enumerate(partition_counts["rows"]):
    axis.annotate(f"{int(count):,}", (position, count),
                  xytext=(0, 4), textcoords="offset points", ha="center")
figure.tight_layout(rect=(0, 0.09, 1, 1))
save_audit_figure(
    figure, "figure_01_partition_sizes.png",
    "Figure 1  Audited encounter counts by protocol partition",
)

figure, axis = plt.subplots(figsize=(7.2, 4.4))
axis.bar(
    ["Adverse outcome", "30-day ED return"],
    [item["positive_fraction"] * 100 for item in label_records],
    color=["#57738C", "#BD8D4B"],
    edgecolor="black", linewidth=0.6
)
axis.set_ylabel("Positive encounters (%)")
axis.set_ylim(0, 100)
for position, item in enumerate(label_records):
    value = item["positive_fraction"] * 100
    axis.annotate(f"{value:.2f}%", (position, value),
                  xytext=(0, 4), textcoords="offset points", ha="center")
figure.tight_layout(rect=(0, 0.09, 1, 1))
save_audit_figure(
    figure, "figure_02_development_outcomes.png",
    "Figure 2  Outcome prevalence within development data D",
)

top_missing = missing_profile.head(15).iloc[::-1]
figure, axis = plt.subplots(figsize=(8.4, 6.8))
axis.barh(
    top_missing["column"], top_missing["missing_fraction"] * 100,
    color="#57738C", edgecolor="black", linewidth=0.5
)
axis.set_xlabel("Missing values within D (%)")
axis.set_xlim(0, 100)
figure.tight_layout(rect=(0, 0.08, 1, 1))
save_audit_figure(
    figure, "figure_03_development_missingness.png",
    "Figure 3  Fields with the highest missingness in development data",
)
save_json("plot_metadata.json", {
    "font_family": resolved_font,
    "captions": "below",
    "source_partition_for_targets_and_missingness": "development",
})

## 本次执行结果

In [ ]:
summary = {
    "status": "completed",
    "dataset_kind": config["dataset_kind"],
    "release_id": release_id,
    "development_rows": len(development),
    "temporal_test_rows": len(temporal_test),
    "supplemental_rows": len(supplemental),
    "kaggle_test_rows": len(test),
    "patient_id_column": patient_column,
    "grouping_unit": "patient" if patient_column else "encounter",
    "source_schema_differences": {
        "train_only": sorted(train_columns - test_columns),
        "test_only": sorted(test_columns - train_columns),
    },
    "audit_directory": str(audit_directory),
    "protocol_version": config["protocol_version"],
}
save_json("summary.json", summary)
print(json.dumps(summary, ensure_ascii=False, indent=2))